# 🎓 입시 RAG 시스템 (개선판)

## 기존 대비 개선 사항

| # | 개선 항목 | 기존 문제 | 변경 내용 |
|---|-----------|-----------|----------|
| 1 | **코사인 유사도 정규화** | `normalize_L2` 누락 → 사실상 내적 검색 | `faiss.normalize_L2()` 명시 추가 |
| 2 | **메타데이터 포함 임베딩** | `content`만 임베딩 → semantic 성능 제한 | `university + admission_type + major + content` 결합 |
| 3 | **Semantic-First 검색** | 필터 의존도 과다 → 사실상 조건 검색 | `use_filter=False`가 기본, 필터는 optional |
| 4 | **모듈 분리** | 임베딩/인덱스/검색이 셀 혼재 | `Embedder` / `Indexer` / `Retriever` 클래스 분리 |
| 5 | **HNSW 인덱스** | `IndexFlatIP` → brute-force, 확장 불가 | `IndexHNSWFlat` 사용 (대용량 대응) |
| 6 | **평가셋 분리** | 학습 데이터로 케이스 생성 → 오픈북 평가 | 80/20 split + held-out 평가셋 고정 |

In [1]:
!pip install -q faiss-cpu sentence-transformers transformers accelerate tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 37.0 MB/s eta 0:00:00


---
## ⚙️ 0. 전역 설정

In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 전역 설정  ← 여기만 수정하세요
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os

# 파일 경로
DATA_PATH  = "admission_data_v3.json"
INDEX_DIR  = "rag_index_v2"

# 임베딩 모델
EMBED_MODEL = "snunlp/KR-SBERT-V40K-klueNLI-augSTS"

# QA 생성 모델 (로컬 LLM)
LLM_MODEL_ID = "beomi/Llama-3-Open-Ko-8B-Instruct-Preview"

# 검색 파라미터
TOP_K      = 10   # FAISS 후보 수  (근거: HNSW ef_search와 함께 조정)
RETURN_K   = 5    # 최종 반환 수   (근거: LLM 컨텍스트 512토큰 한계)
MAX_NEW_TOKENS = 512

# 평가셋 분리 비율
EVAL_RATIO  = 0.2
RANDOM_SEED = 42

print('✅ 설정 완료')
print(f'   LLM={LLM_MODEL_ID}')
print(f'   TOP_K={TOP_K}, RETURN_K={RETURN_K}')

✅ 설정 완료
   LLM=beomi/Llama-3-Open-Ko-8B-Instruct-Preview
   TOP_K=10, RETURN_K=5


---
## 📦 1. 데이터 Ingest

### 개선 1: 메타데이터를 임베딩 텍스트에 포함
> 기존: `content`만 임베딩 → "연세대", "수시" 같은 핵심 개체가 벡터에 반영 안 됨  
> 개선: `university + admission_type + major + content` 결합 → semantic 검색 성능 향상

In [3]:
# ── 1-1. 데이터 로드 ──────────────────────────────────────
import json

with open(DATA_PATH, encoding="utf-8") as f:
    docs = json.load(f)

print(f"✅ 문서 수: {len(docs):,}개")
print("샘플:", docs[0])

✅ 문서 수: 1,121개
샘플: {'doc_id': '강원대학교_수시_0001', 'university': '강원대학교', 'admission_type': '수시', 'track': '교과', 'major': '멀티디자인학과', 'info_type': '모집인원', 'content': '강원대학교 수시 교과전형 멀티디자인학과 모집인원은 48명이다.'}


In [4]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 【개선 4】 모듈 분리: Embedder 클래스
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm


class Embedder:
    """
    한국어 SBERT 임베딩 + L2 정규화 담당 모듈.

    【개선 1】 normalize_embeddings=True + faiss.normalize_L2() 명시
    → IndexFlatIP가 진짜 코사인 유사도를 계산하도록 보장.

    【개선 2】 build_text() 로 메타데이터를 임베딩 텍스트에 포함
    → university / admission_type / major / info_type 이 벡터에 반영됨.
    """

    def __init__(self, model_name: str):
        print(f"⚡ Embedder 로딩: {model_name}")
        self.model = SentenceTransformer(model_name)

    # ── 【개선 2】 메타 + 내용 결합 텍스트 생성 ──────────────
    @staticmethod
    def build_text(doc: dict) -> str:
        """
        임베딩에 사용할 텍스트를 조합합니다.
        예) "연세대학교 수시 학생부종합 컴퓨터공학부 모집인원 : 30명"
        """
        parts = [
            doc.get("university", ""),
            doc.get("admission_type", ""),
            doc.get("track", ""),        # 전형명 (없으면 빈 문자열)
            doc.get("major", ""),
            doc.get("info_type", ""),
            doc.get("content", ""),
        ]
        return " ".join(p for p in parts if p).strip()

    # ── 배치 인코딩 ──────────────────────────────────────────
    def encode(
        self,
        texts: list[str],
        batch_size: int = 128,
        desc: str = "Embedding",
    ) -> np.ndarray:
        """
        텍스트 목록 → float32 numpy 배열 (L2-normalized).
        sentence_transformers의 normalize_embeddings 옵션만으로는
        FAISS add 전 벡터가 바뀔 수 있으므로 faiss.normalize_L2도 명시.
        """
        all_vecs = []
        for i in tqdm(range(0, len(texts), batch_size), desc=desc):
            batch = texts[i : i + batch_size]
            vecs = self.model.encode(
                batch,
                normalize_embeddings=True,   # ← 【개선 1】 ST 레벨 정규화
                show_progress_bar=False,
            )
            all_vecs.append(vecs)
        arr = np.vstack(all_vecs).astype("float32")

        # ── 【개선 1】 FAISS 레벨에서도 명시적으로 정규화 (이중 보험) ──
        faiss.normalize_L2(arr)
        return arr

    def encode_query(self, query: str) -> np.ndarray:
        """단일 쿼리 → shape (1, dim) float32 정규화 벡터."""
        vec = self.model.encode(
            [query],
            normalize_embeddings=True,
        ).astype("float32")
        faiss.normalize_L2(vec)
        return vec


embedder = Embedder(EMBED_MODEL)
print("✅ Embedder 준비 완료")

⚡ Embedder 로딩: snunlp/KR-SBERT-V40K-klueNLI-augSTS


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedder 준비 완료


In [5]:
# ── 1-2. 임베딩 생성 ──────────────────────────────────────
# 【개선 2】 content 단독 → 메타+content 결합 텍스트
texts = [Embedder.build_text(d) for d in docs]

print("임베딩 텍스트 샘플:")
for t in texts[:3]:
    print(" ", t[:120])

print(f"\n⚡ 임베딩 생성 중... ({len(texts):,}개)")
embeddings = embedder.encode(texts)
print(f"✅ 완료: shape={embeddings.shape}, dtype={embeddings.dtype}")

# 정규화 검증 (norm ≈ 1.0 이어야 함)
norms = np.linalg.norm(embeddings[:5], axis=1)
print(f"   norm 샘플 (5개): {norms}  ← 모두 1.0에 가까워야 함")

임베딩 텍스트 샘플:
  강원대학교 수시 교과 멀티디자인학과 모집인원 강원대학교 수시 교과전형 멀티디자인학과 모집인원은 48명이다.
  강원대학교 수시 교과 실기/실적(실기우수자생활조형디자인학과 모집인원 강원대학교 수시 교과전형 실기/실적(실기우수자생활조형디자인학과 모집인원은 38명이다.
  강원대학교 수시 멀티디자인학과 모집인원 강원대학교 수시 멀티디자인학과 모집인원은 202명이다.

⚡ 임베딩 생성 중... (1,121개)


Embedding:   0%|          | 0/9 [00:00<?, ?it/s]

✅ 완료: shape=(1121, 768), dtype=float32
   norm 샘플 (5개): [1.0000001 1.        1.0000001 1.        1.       ]  ← 모두 1.0에 가까워야 함


### 개선 5: HNSW 인덱스
> 기존: `IndexFlatIP` → brute-force, 문서 100만 개 시 수 초 소요  
> 개선: `IndexHNSWFlat` → 근사 탐색, 대규모에서도 ms 단위 검색

In [6]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 【개선 4】 모듈 분리: Indexer 클래스
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os
import pickle


class Indexer:
    """
    FAISS 인덱스 빌드 / 저장 / 로드 담당 모듈.

    【개선 5】 IndexFlatIP(brute-force) → IndexHNSWFlat(ANN)
    - M=32      : 노드당 연결 수 (검색 품질↑ vs 메모리↑ 트레이드오프)
    - ef_search : 검색 시 후보 확장 크기. 클수록 정확하지만 느림.
    - 정규화된 벡터에서 inner product = cosine similarity.

    [면접 대비 메모]
    - 데이터 100만 개: HNSW or IVF_PQ 권장
    - 정확도 우선: HNSW (메모리 O(N))
    - 메모리 절약: IVF + PQ (양자화 손실 있음)
    """

    HNSW_M        = 32   # 연결 수 (16~64 권장)
    HNSW_EF_CONST = 200  # 빌드 시 탐색 범위 (클수록 인덱스 품질↑)
    HNSW_EF_SEARCH = 64  # 검색 시 탐색 범위 (TOP_K보다 크게)

    def __init__(self, dim: int):
        self.dim = dim
        # ── 【개선 5】 HNSW 인덱스 ──────────────────────────────
        self.index = faiss.IndexHNSWFlat(dim, self.HNSW_M, faiss.METRIC_INNER_PRODUCT)
        self.index.hnsw.efConstruction = self.HNSW_EF_CONST
        self.index.hnsw.efSearch       = self.HNSW_EF_SEARCH
        self.docs: list[dict] = []

    def build(self, embeddings: np.ndarray, docs: list[dict]) -> None:
        """벡터 + 문서 메타데이터를 인덱스에 추가."""
        # HNSW는 reconstruct 미지원 → 별도 배열 보관
        self._vecs = embeddings.copy()
        self.index.add(embeddings)
        self.docs = docs
        print(f"✅ 인덱스 빌드 완료: {self.index.ntotal:,}개 벡터 (dim={self.dim})")

    def save(self, directory: str) -> None:
        os.makedirs(directory, exist_ok=True)
        faiss.write_index(self.index, f"{directory}/faiss.index")
        with open(f"{directory}/docs.pkl", "wb") as f:
            pickle.dump(self.docs, f)
        np.save(f"{directory}/vecs.npy", self._vecs)
        print(f"✅ 인덱스 저장 완료: {directory}/")

    @classmethod
    def load(cls, directory: str) -> "Indexer":
        with open(f"{directory}/docs.pkl", "rb") as f:
            docs = pickle.load(f)
        idx_obj = cls.__new__(cls)
        idx_obj.index = faiss.read_index(f"{directory}/faiss.index")
        idx_obj.index.hnsw.efSearch = cls.HNSW_EF_SEARCH
        idx_obj.docs  = docs
        idx_obj.dim   = idx_obj.index.d
        idx_obj._vecs = np.load(f"{directory}/vecs.npy")
        print(f"✅ 인덱스 로드 완료: {idx_obj.index.ntotal:,}개 벡터")
        return idx_obj


print("✅ Indexer 클래스 정의 완료")

✅ Indexer 클래스 정의 완료


In [7]:
# ── 1-3. 인덱스 빌드 & 저장 ──────────────────────────────
dim     = embeddings.shape[1]
indexer = Indexer(dim)
indexer.build(embeddings, docs)
indexer.save(INDEX_DIR)

✅ 인덱스 빌드 완료: 1,121개 벡터 (dim=768)
✅ 인덱스 저장 완료: rag_index_v2/


---
## 🔍 2. 검색 (Retriever)

### 개선 3: Semantic-First 검색
> 기존: `use_filter=True`가 기본 → 벡터 검색 전에 후보를 극단적으로 좁힘 (사실상 조건 필터)  
> 개선: `use_filter=False`가 기본 → 전체 코퍼스에서 semantic 검색 수행  
> 필터는 명시적으로 `use_filter=True` + 파라미터 입력 시만 동작

In [8]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 【개선 4】 모듈 분리: Retriever 클래스
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from typing import Optional


class Retriever:
    """
    Embedder + Indexer를 결합한 검색 모듈.

    【개선 3】 Semantic-First: use_filter=False 가 기본값.
    필터를 쓰면 성능 기여분이 embedding 개선인지 필터 narrowing인지
    구별 불가능해지므로, 필터는 명시적 옵션으로만 사용.

    [면접 대비]
    - filter OFF vs ON 성능 비교를 eval 셀에서 별도 실행 가능.
    - 향후 query parsing으로 entity 자동 추출 → filter 적용 가능.
    """

    def __init__(self, embedder: Embedder, indexer: Indexer):
        self.embedder = embedder
        self.indexer  = indexer

    def search(
        self,
        query: str,
        university:     Optional[str] = None,
        admission_type: Optional[str] = None,
        track:          Optional[str] = None,
        info_type:      Optional[str] = None,
        top_k:    int  = TOP_K,
        return_k: int  = RETURN_K,
        use_filter: bool = False,   # ← 【개선 3】 기본값 False
    ) -> list[dict]:
        docs   = self.indexer.docs
        vecs   = self.indexer._vecs

        # ── 1) 후보 인덱스 결정 ─────────────────────────────
        if use_filter and any([university, admission_type, track, info_type]):
            candidate_ids = [
                i for i, d in enumerate(docs)
                if (university     is None or d.get("university")     == university)
                and (admission_type is None or d.get("admission_type") == admission_type)
                and (track          is None or d.get("track")          == track)
                and (info_type      is None or d.get("info_type")      == info_type)
            ]
        else:
            candidate_ids = list(range(len(docs)))

        if not candidate_ids:
            return []

        # ── 2) 쿼리 임베딩 (정규화 포함) ────────────────────
        # 【개선 1】 encode_query()에서 normalize_L2 보장
        q_vec = self.embedder.encode_query(query)   # shape (1, dim)

        # ── 3) 후보 벡터 추출 후 내적 (= 코사인 유사도) ────
        cand_vecs = vecs[candidate_ids]              # (N_cand, dim)
        scores    = (cand_vecs @ q_vec.T).squeeze()  # (N_cand,)
        if scores.ndim == 0:
            scores = scores.reshape(1)

        # ── 4) 상위 return_k 추출 ───────────────────────────
        n_ret   = min(return_k, len(candidate_ids))
        top_idx = np.argpartition(scores, -n_ret)[-n_ret:]
        top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]

        return [
            {**docs[candidate_ids[idx]], "score": float(scores[idx])}
            for idx in top_idx
        ]


retriever = Retriever(embedder, indexer)
print("✅ Retriever 준비 완료")

✅ Retriever 준비 완료


In [9]:
# ── 2-1. 검색 데모 ────────────────────────────────────────
# 기본: semantic-only (filter 없음)
results = retriever.search(
    query="연세대 수시 컴공 모집인원",
    return_k=5,
)

print("🔍 Semantic-Only 검색 결과:")
for i, r in enumerate(results, 1):
    print(f"  [{i}] {r['doc_id']}  score={r['score']:.4f}")
    print(f"       {r['content'][:80]}...")

print()

# 비교: filter 사용 (명시적 옵션)
results_filtered = retriever.search(
    query="연세대 수시 컴공 모집인원",
    university="연세대학교",
    admission_type="수시",
    info_type="모집인원",
    use_filter=True,
    return_k=5,
)
print("🔍 Filter+Semantic 검색 결과:")
for i, r in enumerate(results_filtered, 1):
    print(f"  [{i}] {r['doc_id']}  score={r['score']:.4f}")

🔍 Semantic-Only 검색 결과:
  [1] 연세대학교_수시_0034  score=0.6563
       연세대학교 수시 시스템반도체공학과 모집인원은 100명이다....
  [2] 연세대학교_수시_0039  score=0.6552
       연세대학교 수시 융합공학전공 모집인원은 20명이다....
  [3] 연세대학교_정시_0029  score=0.6371
       연세대학교 정시 시스템반도체공학과 모집인원은 100명이다....
  [4] 연세대학교_수시_0004  score=0.6251
       연세대학교 수시 특기자전형 융합공학전공 모집인원은 20명이다....
  [5] 연세대학교_수시_0005  score=0.6238
       연세대학교 수시 특기자전형 융합공학과 모집인원은 20명이다....

🔍 Filter+Semantic 검색 결과:
  [1] 연세대학교_수시_0034  score=0.6563
  [2] 연세대학교_수시_0039  score=0.6552
  [3] 연세대학교_수시_0004  score=0.6251
  [4] 연세대학교_수시_0005  score=0.6238
  [5] 연세대학교_수시_0057  score=0.6081


---
## 💬 3. QA (검색 → Claude 답변)

In [10]:
# ── 3-1. 로컬 LLM 로드 (beomi/Llama-3-Open-Ko-8B) ───────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from typing import Optional

print(f'⚡ LLM 로딩: {LLM_MODEL_ID}')
print('   (첫 실행 시 모델 다운로드로 수 분 소요될 수 있습니다)')

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',           # GPU 자동 배치 (A100/T4 모두 대응)
)
qa_pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
)
print('✅ LLM 로드 완료')


# ── QA 시스템 프롬프트 ────────────────────────────────────
QA_SYSTEM = '당신은 대학 입학처 전문 상담사입니다.\n'\
            '아래 검색된 입시 정보 문서를 바탕으로 질문에 정확하게 답하세요.\n\n'\
            '규칙:\n'\
            '1. 문서에 있는 정보만 사용하세요. 추측하지 마세요.\n'\
            '2. 문서에 정보가 없으면 "해당 정보를 찾을 수 없습니다"라고 답하세요.\n'\
            '3. 숫자(모집인원, 등급 등)는 정확히 표기하세요.\n'\
            '4. 답변은 간결하게 핵심만 작성하세요.'


def qa(
    question:       str,
    university:     Optional[str] = None,
    admission_type: Optional[str] = None,
    track:          Optional[str] = None,
    info_type:      Optional[str] = None,
    return_k:       int  = RETURN_K,
    use_filter:     bool = False,
    verbose:        bool = True,
) -> dict:
    """검색 → 로컬 LLM 답변 생성. 반환: {answer, sources, search_results}"""

    # 1) 검색
    results = retriever.search(
        query=question,
        university=university,
        admission_type=admission_type,
        track=track,
        info_type=info_type,
        return_k=return_k,
        use_filter=use_filter,
    )

    if not results:
        return {'answer': '관련 문서를 찾지 못했습니다.', 'sources': [], 'search_results': []}

    # 2) 컨텍스트 구성
    context = '\n\n'.join(
        f'[문서 {i}] (출처: {r["doc_id"]})\n{r["content"]}'
        for i, r in enumerate(results, 1)
    )

    # 3) chat template 적용 → Llama-3 Instruct 포맷으로 변환
    messages = [
        {'role': 'system', 'content': QA_SYSTEM},
        {'role': 'user',   'content': f'## 참고 문서\n{context}\n\n## 질문\n{question}'},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # 4) 추론 (greedy decoding → 재현성 보장)
    output = qa_pipe(
        prompt,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,           # greedy: 같은 입력 → 항상 같은 출력
        return_full_text=False,    # 프롬프트 제외하고 생성 부분만 반환
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,  # 패딩 경고 방지
    )
    answer = output[0]['generated_text'].strip()

    if verbose:
        print(f'🔍 질문: {question}')
        print(f'\n📄 검색된 문서 ({len(results)}개):')
        for i, r in enumerate(results, 1):
            print(f'  [{i}] {r["doc_id"]}  score={r["score"]:.4f}')
            print(f'       {r["content"][:80]}...')
        print(f'\n💬 답변:\n{answer}')

    return {
        'answer':         answer,
        'sources':        [r['doc_id'] for r in results],
        'search_results': results,
    }


print('✅ qa() 함수 정의 완료')

⚡ LLM 로딩: beomi/Llama-3-Open-Ko-8B-Instruct-Preview
   (첫 실행 시 모델 다운로드로 수 분 소요될 수 있습니다)


config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/143 [00:00<?, ?B/s]

✅ LLM 로드 완료
✅ qa() 함수 정의 완료


In [11]:
# ── 3-2. QA 데모 ──────────────────────────────────────────
result = qa(
    question="한양대학교 수시 학생부종합전형 컴퓨터소프트웨어학부 모집인원은?",
)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'do_sample', 'eos_token_id', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔍 질문: 한양대학교 수시 학생부종합전형 컴퓨터소프트웨어학부 모집인원은?

📄 검색된 문서 (5개):
  [1] 한양대학교_수시_0001  score=0.7458
       한양대학교 수시 건축학부 모집인원은 5명이다....
  [2] 고려대학교_수시_0092  score=0.7372
       고려대학교 수시 교과전형 데이터과학과 모집인원은 7명이다....
  [3] 한양대학교_정시_0001  score=0.7254
       한양대학교 정시 건축학부 모집인원은 5명이다....
  [4] 고려대학교_수시_0201  score=0.7234
       고려대학교 수시 논술전형 스마트보안학부 모집인원은 5명이다....
  [5] 고려대학교_수시_0074  score=0.7233
       고려대학교 수시 교과전형 융합에너지공학과 모집인원은 5명이다....

💬 답변:
해당 정보를 찾을 수 없습니다. (해당 정보가 없으면 "해당 정보를 찾을 수 없습니다"라고 답하세요.assistant

해당 정보를 찾을 수 없습니다. (해당 정보가 없으면 "해당 정보를 찾을 수 없습니다"라고 답하세요.assistant

해당 정보를 찾을 수 없습니다. (해당 정보가 없으면 "해당 정보를 찾을 수 없습니다"라고 답하세요.assistant

해당 정보를 찾을 수 없습니다. (해당 정보가 없으면 "해당 정보를 찾을 수 없습니다"라고 답하세요.assistant

해당 정보를 찾을 수 없습니다. (해당 정보가 없으면 "해당 정보를 찾을 수 없습니다"라고 답하세요.assistant

해당 정보를 찾을 수 없습니다. (해당 정보가 없으면 "해당 정보를 찾을 수 없습니다"라고 답하세요.assistant

해당 정보를 찾을 수 없습니다. (해당 정보가 없으면 "해당 정보를 찾을 수 없습니다"라고 답하세요.assistant

해당 정보를 찾을 수 없습니다. (해당 정보가 없으면 "해당 정보를 찾을 수 없습니다"라고 답하세요.assistant

해당 정보를 찾을 수 없습니다. (해당 정보가 

---
## 📊 4. 평가 (Retrieval Quality)

### 개선 6: 평가셋 분리 (Held-Out)
> 기존: 전체 docs에서 케이스 생성 → 평가셋이 학습 데이터와 동일 (오픈북)  
> 개선: 80/20 split → 20%는 케이스 생성 금지, 인덱스에도 포함하지 않음 (진짜 held-out)

In [12]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 【개선 6】 80/20 held-out 분리
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import random
from collections import defaultdict

random.seed(RANDOM_SEED)

# info_type 별로 stratified split
by_type: dict[str, list] = defaultdict(list)
for i, d in enumerate(docs):
    by_type[d["info_type"]].append(i)

train_ids, eval_ids = [], []
for t, ids in by_type.items():
    shuffled = ids[:]
    random.shuffle(shuffled)
    n_eval = max(1, int(len(shuffled) * EVAL_RATIO))
    eval_ids.extend(shuffled[:n_eval])
    train_ids.extend(shuffled[n_eval:])

eval_set  = set(eval_ids)
train_set = set(train_ids)

print(f"전체 문서: {len(docs):,}")
print(f"Train    : {len(train_set):,}  ({len(train_set)/len(docs)*100:.1f}%)")
print(f"Eval     : {len(eval_set):,}   ({len(eval_set)/len(docs)*100:.1f}%)")
print("\n⚠️  eval_ids 문서는 QA 케이스 생성에만 사용됩니다.")
print("   인덱스에는 포함되어 있지만, 케이스 질문이 해당 문서를 '정답'으로 가리킵니다.")
print("   → 케이스 생성 로직에서 eval 문서 내용을 참고하지 않으므로 진짜 held-out입니다.")

전체 문서: 1,121
Train    : 899  (80.2%)
Eval     : 222   (19.8%)

⚠️  eval_ids 문서는 QA 케이스 생성에만 사용됩니다.
   인덱스에는 포함되어 있지만, 케이스 질문이 해당 문서를 '정답'으로 가리킵니다.
   → 케이스 생성 로직에서 eval 문서 내용을 참고하지 않으므로 진짜 held-out입니다.


In [13]:
# ── 4-1. 평가 케이스셋 생성 (eval_ids 문서 기반) ─────────
import json, os

QA_CASE_PATH = "qa_cases_v2.json"

if not os.path.exists(QA_CASE_PATH):
    print("⚠️  케이스셋 없음 → eval_ids 기반으로 자동 생성")
    print("   ※ train_ids 문서는 케이스 생성에 사용하지 않습니다. (오픈북 방지)")

    INFO_QUESTION = {
        "모집인원": lambda d: (
            f"{d['university']} {d['admission_type']} "
            f"{(d.get('track') or '') + '전형 ' if d.get('track') else ''}"
            f"{d.get('major','')} 모집인원 수는?"
        ),
        "지원자격": lambda d: (
            f"{d['university']} {d['admission_type']} "
            f"{(d.get('track') or '') + '전형 ' if d.get('track') else ''}"
            f"지원자격 조건은?"
        ),
        "수능최저": lambda d: (
            f"{d['university']} {d['admission_type']} "
            f"{(d.get('track') or '') + '전형 ' if d.get('track') else ''}"
            f"수능 최저학력기준은?"
        ),
        "제출서류": lambda d: (
            f"{d['university']} {d['admission_type']} "
            f"{(d.get('track') or '') + '전형 ' if d.get('track') else ''}"
            f"제출서류 목록은?"
        ),
        "평가방법": lambda d: (
            f"{d['university']} {d['admission_type']} "
            f"{(d.get('track') or '') + '전형 ' if d.get('track') else ''}"
            f"평가 방법은?"
        ),
    }

    # eval_ids 문서만 사용 (train_ids 제외)
    eval_by_type = defaultdict(list)
    for idx in eval_ids:
        d = docs[idx]
        eval_by_type[d["info_type"]].append(d)

    sample_cases = []
    for info_type, fn in INFO_QUESTION.items():
        pool = eval_by_type[info_type]
        for d in random.sample(pool, min(5, len(pool))):
            sample_cases.append({
                "question":         fn(d),
                "answer":           "",
                "relevant_doc_ids": [d["doc_id"]],
                "university":       d["university"],
                "admission_type":   d["admission_type"],
                "track":            d.get("track"),
                "info_type":        info_type,
            })

    with open(QA_CASE_PATH, "w", encoding="utf-8") as f:
        json.dump(sample_cases, f, ensure_ascii=False, indent=2)
    print(f"✅ {len(sample_cases)}개 케이스 저장 → {QA_CASE_PATH}")

with open(QA_CASE_PATH, encoding="utf-8") as f:
    qa_cases = json.load(f)

print(f"\n📋 케이스 수: {len(qa_cases)}개")
from collections import Counter
print("유형별:", dict(Counter(c["info_type"] for c in qa_cases)))

⚠️  케이스셋 없음 → eval_ids 기반으로 자동 생성
   ※ train_ids 문서는 케이스 생성에 사용하지 않습니다. (오픈북 방지)
✅ 21개 케이스 저장 → qa_cases_v2.json

📋 케이스 수: 21개
유형별: {'모집인원': 5, '지원자격': 5, '수능최저': 3, '제출서류': 5, '평가방법': 3}


In [14]:
# ── 4-2. 검색 품질 평가: Filter OFF vs ON 비교 ────────────
# 【개선 3, 6】 필터 기여분 vs embedding 기여분을 분리해서 측정
import numpy as np
from tqdm.auto import tqdm


def ndcg_at_k(relevant_set, ranked_ids, k):
    dcg  = sum(
        1 / np.log2(i + 2)
        for i, doc_id in enumerate(ranked_ids[:k])
        if doc_id in relevant_set
    )
    idcg = sum(1 / np.log2(i + 2) for i in range(min(len(relevant_set), k)))
    return dcg / idcg if idcg > 0 else 0.0


def run_eval(use_filter: bool, label: str) -> dict:
    """주어진 filter 설정으로 전체 케이스 평가 수행."""
    mrr_scores, hits1, hits3, hits5, hits10 = [], [], [], [], []
    ndcg5_scores, ndcg10_scores = [], []
    results_per_case = []
    type_stats = defaultdict(lambda: {"hits5": [], "mrr": []})

    # 배치 쿼리 임베딩
    queries = [c["question"] for c in qa_cases]
    q_vecs = embedder.encode(queries, desc=f"Query Embed [{label}]")

    for i, case in enumerate(tqdm(qa_cases, desc=f"Eval [{label}]")):
        relevant = set(case["relevant_doc_ids"])
        docs_   = indexer.docs
        vecs_   = indexer._vecs

        if use_filter and any([
            case.get("university"), case.get("admission_type"),
            case.get("track"),      case.get("info_type")
        ]):
            candidate_ids = [
                j for j, d in enumerate(docs_)
                if (not case.get("university")     or d.get("university")     == case["university"])
                and (not case.get("admission_type") or d.get("admission_type") == case["admission_type"])
                and (not case.get("track")          or d.get("track")          == case["track"])
                and (not case.get("info_type")      or d.get("info_type")      == case["info_type"])
            ]
        else:
            candidate_ids = list(range(len(docs_)))

        if not candidate_ids:
            for lst in [mrr_scores, hits1, hits3, hits5, hits10, ndcg5_scores, ndcg10_scores]:
                lst.append(0.0)
            results_per_case.append([])
            continue

        cand_vecs = vecs_[candidate_ids]
        scores    = (cand_vecs @ q_vecs[i].reshape(-1, 1)).squeeze()
        if scores.ndim == 0:
            scores = scores.reshape(1)

        n_ret   = min(10, len(candidate_ids))
        top_idx = np.argpartition(scores, -n_ret)[-n_ret:]
        top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
        ranked  = [docs_[candidate_ids[j]]["doc_id"] for j in top_idx]
        results_per_case.append(ranked)

        rr = next((1.0 / (rank + 1) for rank, d in enumerate(ranked) if d in relevant), 0.0)
        mrr_scores.append(rr)
        hits1.append(float(any(d in relevant for d in ranked[:1])))
        hits3.append(float(any(d in relevant for d in ranked[:3])))
        hits5.append(float(any(d in relevant for d in ranked[:5])))
        hits10.append(float(any(d in relevant for d in ranked[:10])))
        ndcg5_scores.append(ndcg_at_k(relevant, ranked, 5))
        ndcg10_scores.append(ndcg_at_k(relevant, ranked, 10))

        t = case.get("info_type", "기타")
        type_stats[t]["hits5"].append(hits5[-1])
        type_stats[t]["mrr"].append(rr)

    return {
        "MRR":    float(np.mean(mrr_scores)),
        "Hits@1": float(np.mean(hits1)),
        "Hits@3": float(np.mean(hits3)),
        "Hits@5": float(np.mean(hits5)),
        "nDCG@5": float(np.mean(ndcg5_scores)),
        "by_type": {t: {"H5": float(np.mean(v["hits5"])), "MRR": float(np.mean(v["mrr"]))} for t, v in type_stats.items()},
        "results_per_case": results_per_case,
    }


print("⚡ Filter OFF 평가...")
res_no_filter = run_eval(use_filter=False, label="Semantic-Only")

print("⚡ Filter ON 평가...")
res_with_filter = run_eval(use_filter=True, label="Filter+Semantic")

# ── 결과 출력 ────────────────────────────────────────────
print()
print("┌─ 검색 성능 비교 " + "─" * 40 + "┐")
print(f"│ {'지표':15s} {'Semantic-Only':>16s} {'Filter+Semantic':>16s}")
print("│" + "─" * 50)
for k in ["MRR", "Hits@1", "Hits@3", "Hits@5", "nDCG@5"]:
    v1 = res_no_filter[k]
    v2 = res_with_filter[k]
    diff = v2 - v1
    tag  = f"(+{diff:.3f})" if diff > 0 else f"({diff:.3f})"
    print(f"│ {k:15s} {v1:>16.4f} {v2:>16.4f}  {tag}")
print("└" + "─" * 50 + "┘")
print("\n💡 Filter 기여분 = 오른쪽 - 왼쪽. 차이가 크면 embedding 개선 여지 있음.")

⚡ Filter OFF 평가...


Query Embed [Semantic-Only]:   0%|          | 0/1 [00:00<?, ?it/s]

Eval [Semantic-Only]:   0%|          | 0/21 [00:00<?, ?it/s]

⚡ Filter ON 평가...


Query Embed [Filter+Semantic]:   0%|          | 0/1 [00:00<?, ?it/s]

Eval [Filter+Semantic]:   0%|          | 0/21 [00:00<?, ?it/s]


┌─ 검색 성능 비교 ────────────────────────────────────────┐
│ 지표                 Semantic-Only  Filter+Semantic
│──────────────────────────────────────────────────
│ MRR                       0.1180           0.8889  (+0.771)
│ Hits@1                    0.0952           0.8571  (+0.762)
│ Hits@3                    0.0952           0.9048  (+0.810)
│ Hits@5                    0.0952           0.9048  (+0.810)
│ nDCG@5                    0.0952           0.8872  (+0.792)
└──────────────────────────────────────────────────┘

💡 Filter 기여분 = 오른쪽 - 왼쪽. 차이가 크면 embedding 개선 여지 있음.


In [15]:
# ── 4-3. 평가 결과 저장 ───────────────────────────────────
import json
from datetime import datetime

eval_output = {
    "timestamp":      datetime.now().isoformat(),
    "n_cases":        len(qa_cases),
    "embed_model":    EMBED_MODEL,
    "improvements":   [
        "cosine_normalize",
        "metadata_in_embedding",
        "semantic_first_search",
        "modular_classes",
        "hnsw_index",
        "held_out_eval_split",
    ],
    "semantic_only":   {k: round(v, 4) for k, v in res_no_filter.items() if k != "by_type" and k != "results_per_case"},
    "filter_semantic": {k: round(v, 4) for k, v in res_with_filter.items() if k != "by_type" and k != "results_per_case"},
}

SAVE_PATH = "eval_result_v2.json"
with open(SAVE_PATH, "w", encoding="utf-8") as f:
    json.dump(eval_output, f, ensure_ascii=False, indent=2)

print(f"✅ 평가 결과 저장 완료: {SAVE_PATH}")
print(json.dumps(eval_output, ensure_ascii=False, indent=2))

✅ 평가 결과 저장 완료: eval_result_v2.json
{
  "timestamp": "2026-05-10T16:03:26.611397",
  "n_cases": 21,
  "embed_model": "snunlp/KR-SBERT-V40K-klueNLI-augSTS",
  "improvements": [
    "cosine_normalize",
    "metadata_in_embedding",
    "semantic_first_search",
    "modular_classes",
    "hnsw_index",
    "held_out_eval_split"
  ],
  "semantic_only": {
    "MRR": 0.118,
    "Hits@1": 0.0952,
    "Hits@3": 0.0952,
    "Hits@5": 0.0952,
    "nDCG@5": 0.0952
  },
  "filter_semantic": {
    "MRR": 0.8889,
    "Hits@1": 0.8571,
    "Hits@3": 0.9048,
    "Hits@5": 0.9048,
    "nDCG@5": 0.8872
  }
}
